# Information Retrieval and Web Analytics

# Final Project - Part 2: Indexing and Evaluation

In this notebook we:

1. **Build the inverted index** of the processed corpus of Part 1 and search with **conjunctive (AND) queries**.
2. **Propose five test queries** and **rank the results with TF-IDF**.
3. **Evaluate** the search engine with P@K, R@K, AP@K, F1@K, MAP, MRR and NDCG, on the queries of
   `validation_labels.csv` and on our own queries (labelled by us in `own_queries_labels.csv`).

The code of the index and the metrics is in `myapp/search/algorithms.py` and `myapp/evaluation/metrics.py`.

### 1 - Packages

In [1]:
import os
import sys
import time
import pickle
from collections import Counter

import numpy as np
import pandas as pd

# the notebook is in project_progress/part_2, so the root of the project is two folders up
ROOT = "../.." if not os.path.exists("myapp") else "."
sys.path.append(ROOT)

from myapp.preprocessing.text_processing import build_terms
from myapp.preprocessing.fields import load_raw_corpus, build_processed_corpus, save_processed_corpus
from myapp.search.algorithms import create_index_tfidf, conjunctive_search, rank_documents, search_tf_idf, tf_idf_weight
from myapp.evaluation.metrics import (precision_at_k, recall_at_k, avg_precision_at_k,
                                      f1_at_k, rr_at_k, ndcg_at_k, evaluate_query, mean_metrics)

CORPUS_PATH = os.path.join(ROOT, "data", "processed", "processed_corpus.pkl")
DATA_PATH = os.path.join(ROOT, "data", "fashion_products_dataset.json")
LABELS_PATH = os.path.join(ROOT, "data", "validation_labels.csv")
OWN_LABELS_PATH = os.path.join(ROOT, "project_progress", "part_2", "own_queries_labels.csv")
K = 10
pd.options.display.float_format = "{:.3f}".format

### 2 - Load the processed corpus of Part 1

If `processed_corpus.pkl` does not exist yet, it is created with the Part 1 functions.

In [2]:
if os.path.exists(CORPUS_PATH):
    with open(CORPUS_PATH, "rb") as fp:
        corpus = pickle.load(fp)
else:
    corpus = build_processed_corpus(load_raw_corpus(DATA_PATH))
    save_processed_corpus(corpus, CORPUS_PATH)

print("Number of documents: {}".format(len(corpus)))

Number of documents: 28080


## Part 1 - Indexing

### 3 - Build the inverted index

As in the lab, the index maps each term to the list of documents where it appears (with its positions).
We index the combined field `search_tokens` of Part 1 (title + description + brand + sub_category +
product_details values). At the same time we compute, as in the theory of the course (log base 2):

- $df_t$: number of documents that contain the term, and $idf_t = \log_2(N / df_t)$
- tf-idf weight: $w_{t,d} = (1 + \log_2 f_{t,d}) \cdot idf_t$ if $f_{t,d} > 0$, else 0
- length of each document: $|d| = \sqrt{\sum_t w_{t,d}^2}$ (L2 norm of all its weights)

In [3]:
start_time = time.time()
index, weights, idf, doc_length = create_index_tfidf(corpus)
df = {term: len(postings) for term, postings in index.items()}   # document frequency of each term
print("Total time to create the index: {} seconds".format(np.round(time.time() - start_time, 2)))
print("Number of terms: {}".format(len(index)))

# example: the term "jean" (the stemmed form of "jeans")
print("\ndf('jean') = {}, idf('jean') = {}".format(df["jean"], np.round(idf["jean"], 4)))
print("First postings of 'jean':", index["jean"][:3])
print("Postings of 'jeans' (not stemmed):", index.get("jeans", []))

Total time to create the index: 4.28 seconds
Number of terms: 5733

df('jean') = 2436, idf('jean') = 3.527
First postings of 'jean': [['TSHFHJKSY8FNR7PV', [48]], ['TSHFHJKRPYSHNEMG', [49]], ['TSHFHJKR4AF3G9KY', [48]]]
Postings of 'jeans' (not stemmed): []


### 4 - Conjunctive (AND) queries

A document is returned only if it contains **all** the terms of the query, so we take the intersection of
the posting lists (in the lab it was the union, OR). Example with the two validation queries:

In [4]:
for query in ["women full sleeve sweatshirt cotton", "men slim jeans blue"]:
    terms = build_terms(query)
    print("{} -> {}".format(query, terms))
    for term in terms:
        print("   df({}) = {}".format(term, df[term]))
    print("   documents with ALL the terms: {}".format(len(conjunctive_search(terms, index))))

women full sleeve sweatshirt cotton -> ['women', 'full', 'sleev', 'sweatshirt', 'cotton']
   df(women) = 13773
   df(full) = 7176
   df(sleev) = 17911
   df(sweatshirt) = 1417
   df(cotton) = 19881
   documents with ALL the terms: 484
men slim jeans blue -> ['men', 'slim', 'jean', 'blue']
   df(men) = 14252
   df(slim) = 5665
   df(jean) = 2436
   df(blue) = 7174
   documents with ALL the terms: 221


### 5 - Propose five test queries

We look at the most popular terms of the titles (document frequency) to choose queries that use frequent
attributes (gender, color, neck type, pattern, product type). We combine easy queries with very common
attributes and harder ones that we expect to fail, so that the evaluation shows the problems of the system.

In [5]:
title_df = Counter()
for doc in corpus.values():
    title_df.update(set(doc["tokens"]["title"]))
print("Most frequent title terms (df):", title_df.most_common(30))

Most frequent title terms (df): [('women', 13041), ('men', 13021), ('tshirt', 12247), ('neck', 11952), ('solid', 9215), ('round', 8301), ('print', 8106), ('pack', 4787), ('blue', 4282), ('fit', 3923), ('shirt', 3591), ('black', 3501), ('casual', 2927), ('slim', 2828), ('polo', 2668), ('multicolor', 2370), ('collar', 2360), ('white', 2168), ('sleev', 2030), ('full', 2011), ('grey', 1913), ('regular', 1754), ('cotton', 1647), ('sweatshirt', 1343), ('dark', 1333), ('design', 1324), ('self', 1320), ('color', 1295), ('block', 1282), ('green', 1276)]


In [6]:
own_queries = {3: "women black round neck tshirt",
               4: "men checkered casual cotton shirt",
               5: "women printed kurta",
               6: "men white sneakers",
               7: "women v neck tshirt"}

for q_id, query in own_queries.items():
    terms = build_terms(query)
    print("Query {}: '{}' -> {} | df: {}".format(q_id, query, terms, [df.get(t, 0) for t in terms]))

Query 3: 'women black round neck tshirt' -> ['women', 'black', 'round', 'neck', 'tshirt'] | df: [13773, 6204, 9511, 14586, 12918]
Query 4: 'men checkered casual cotton shirt' -> ['men', 'checker', 'casual', 'cotton', 'shirt'] | df: [14252, 1248, 9719, 19881, 5125]
Query 5: 'women printed kurta' -> ['women', 'print', 'kurta'] | df: [13773, 9502, 1196]
Query 6: 'men white sneakers' -> ['men', 'white', 'sneaker'] | df: [14252, 3916, 775]
Query 7: 'women v neck tshirt' -> ['women', 'neck', 'tshirt'] | df: [13773, 14586, 12918]


### 6 - Rank the results with TF-IDF (vector model, cosine similarity)

The query is weighted with the same formula as the documents, $w_{t,q} = (1 + \log_2 f_{t,q}) \cdot idf_t$,
and the score is the cosine similarity:
$cos(q, d) = \frac{\sum_t w_{t,q} \, w_{t,d}}{|q| \, |d|}$.
Documents with the same score are sorted by pid. First we check the implementation with the two examples
solved in class (log base 2): the query "to do" (ranking d1 0.660 > d2 0.408 > d3 0.118 > d4 0.058, without
dividing by |q|) and the query "sol playa" (cosine 0.982, 0.735, 0.147, 0).

In [7]:
def check_example(documents, query):
    '''
    Rank a small example collection with our index and cosine functions (no pre-processing).

    Argument:
    documents -- dict doc_id -> text
    query -- query text

    Returns:
    list of (doc_id, cosine, score without dividing by |q|)
    '''
    small = {d: {"search_tokens": text.split()} for d, text in documents.items()}
    ex_index, ex_weights, ex_idf, ex_length = create_index_tfidf(small)
    terms = query.split()
    q_length = np.sqrt(sum(tf_idf_weight(terms.count(t), ex_idf[t]) ** 2 for t in set(terms)))
    ranking = rank_documents(terms, list(documents), ex_weights, ex_idf, ex_length)
    return [(d, float(np.round(s, 3)), float(np.round(s * q_length, 3))) for d, s in ranking]

print("'to do'    :", check_example({"d1": "to do is to be to be is to do", "d2": "to be or not to be i am what i am",
                     "d3": "i think therefore i am do be do be do", "d4": "do do do da da da let it be let it be"}, "to do"))
print("'sol playa':", check_example({"d1": "sol playa sol", "d2": "playa montaña", "d3": "montaña nieve nieve nieve",
                     "d4": "sol nieve playa"}, "sol playa"))

'to do'    : [('d1', 0.609, 0.66), ('d2', 0.377, 0.408), ('d3', 0.109, 0.118), ('d4', 0.053, 0.058)]
'sol playa': [('d1', 0.982, 1.063), ('d4', 0.735, 0.795), ('d2', 0.147, 0.159), ('d3', 0.0, 0.0)]


In [8]:
results = {}
for q_id, query in own_queries.items():
    results[q_id] = search_tf_idf(query, index, weights, idf, doc_length)
    print("\n======================\nQuery {}: '{}' -> {} results. Top 5:".format(q_id, query, len(results[q_id])))
    for rank, (pid, score) in enumerate(results[q_id][:5], 1):
        print("   {}. {} | score = {} | {}".format(rank, pid, np.round(score, 3), corpus[pid]["meta"]["title"]))


Query 3: 'women black round neck tshirt' -> 1112 results. Top 5:
   1. TSHFKJET3YVMZCAS | score = 0.818 | Solid Women Round Neck Black T-Shirt
   2. TSHFUMN8QBZUFHDG | score = 0.748 | Solid Women Round Neck Black T-Shirt
   3. TSHFQF6TNKNDGSYU | score = 0.734 | Solid Women Round Neck Black T-Shirt
   4. TSHFZBYXABMBDQE6 | score = 0.734 | Solid Women Round Neck Black T-Shirt
   5. TSHFNYTT4WNPPPVQ | score = 0.726 | Solid Women Round Neck Black T-Shirt

Query 4: 'men checkered casual cotton shirt' -> 346 results. Top 5:
   1. SHTFWP8CA7WRZBAH | score = 0.783 | Men Regular Fit Checkered Casual Shirt
   2. SHTFWP8CMWVNBYKP | score = 0.783 | Men Regular Fit Checkered Casual Shirt
   3. SHTFSH5WFZFMWPKE | score = 0.775 | Men Regular Fit Checkered Casual Shirt
   4. SHTFP8MFY3CTWA8K | score = 0.768 | Men Regular Fit Checkered Casual Shirt
   5. SHTFWP8CCVETBFZA | score = 0.76 | Men Regular Fit Checkered Casual Shirt

Query 5: 'women printed kurta' -> 42 results. Top 5:
   1. KTAFZCK5RGNECHHH


Query 7: 'women v neck tshirt' -> 6351 results. Top 5:
   1. TSHFKJET3YVMZCAS | score = 0.449 | Solid Women Round Neck Black T-Shirt
   2. TSHFYY7GTESPJSV6 | score = 0.433 | Solid Women Round Neck White T-Shirt
   3. TSHEGFVK5KYTYVHN | score = 0.425 | Solid Women Round Neck White T-Shirt
   4. TSHFNXNJTZDWYMZK | score = 0.422 | Solid Women V Neck Black T-Shirt
   5. TSHFUMN8QBZUFHDG | score = 0.411 | Solid Women Round Neck Black T-Shirt


## Part 2 - Evaluation

### 7 - Evaluation metrics

The metrics are implemented in `myapp/evaluation/metrics.py`. All of them receive the relevance labels sorted
by our ranking. We check them with the examples of the slides (Average Precision and MAP):

In [9]:
def labels_from_positions(positions, n=10):
    '''Binary label array with 1 at the given (1-based) positions.'''
    labels = np.zeros(n, dtype=int)
    labels[[p - 1 for p in positions]] = 1
    return labels

ranking_1 = labels_from_positions([1, 3, 4, 5, 6, 10])   # slide "Example: Average Precision", ranking #1
ranking_2 = labels_from_positions([2, 5, 6, 7, 9, 10])   # ranking #2
print("AP ranking #1: {} (slide: 0.78)".format(np.round(avg_precision_at_k(ranking_1, 10), 3)))
print("AP ranking #2: {} (slide: 0.52)".format(np.round(avg_precision_at_k(ranking_2, 10), 3)))

query_1 = labels_from_positions([1, 3, 6, 9, 10])        # slide "Example: MAP"
query_2 = labels_from_positions([2, 5, 7])
print("MAP: {} (slide: 0.53)".format(np.round(np.mean([avg_precision_at_k(query_1, 10), avg_precision_at_k(query_2, 10)]), 3)))

AP ranking #1: 0.775 (slide: 0.78)
AP ranking #2: 0.521 (slide: 0.52)
MAP: 0.533 (slide: 0.53)


### 8 - Evaluation with `validation_labels.csv`

Only 20 documents are judged for each query, so (as in the Evaluation lab) we rank these 20 documents with
their tf-idf score (ties by pid). A judged document that does not contain all the query terms is **not
returned** by the AND search, so it is not part of the ranking; if it is relevant, it still counts in the
total number of relevant documents (it is a relevant document that the system missed). K = 10.

In [10]:
labels = pd.read_csv(LABELS_PATH)
validation_queries = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}

def judged_ranking(q_id, query, labels_df):
    '''
    Rank the judged documents of a query with tf-idf (0 if they do not contain all the query terms).

    Argument:
    q_id -- query id in labels_df
    query -- query text
    labels_df -- DataFrame with columns pid, query_id, labels

    Returns:
    DataFrame of the judged documents sorted by score (ties by pid), with columns pid, labels, score,
    retrieved (only the rows with retrieved = True are part of the ranking)
    '''
    judged = labels_df[labels_df["query_id"] == q_id][["pid", "labels"]].copy()
    scores = dict(search_tf_idf(query, index, weights, idf, doc_length))
    judged["score"] = [scores.get(pid, 0.0) for pid in judged["pid"]]
    judged["retrieved"] = [pid in scores for pid in judged["pid"]]
    return judged.sort_values(["score", "pid"], ascending=[False, True]).reset_index(drop=True)

validation_rankings = {}
for q_id, query in validation_queries.items():
    validation_rankings[q_id] = judged_ranking(q_id, query, labels)
    r = validation_rankings[q_id]
    print("Query {}: relevant = {}, judged documents returned by the AND search = {}, relevant returned = {}".format(
        q_id, r["labels"].sum(), r["retrieved"].sum(), r[r["retrieved"]]["labels"].sum()))

Query 1: relevant = 13, judged documents returned by the AND search = 20, relevant returned = 13
Query 2: relevant = 10, judged documents returned by the AND search = 8, relevant returned = 6


In [11]:
def evaluation_table(rankings):
    '''
    Compute the metrics of each query and the mean over the queries (MAP, MRR) for a dict q_id -> ranked DataFrame.

    Argument:
    rankings -- dict q_id -> DataFrame sorted by score with the columns "labels" and "retrieved"

    Returns:
    table -- DataFrame with one row per query (P, R, AP, F1, NDCG), rounded to 3 decimals
    means -- dict with MAP@K and MRR@K, rounded to 3 decimals
    '''
    per_query = {}
    for q_id, r in rankings.items():
        returned = r[r["retrieved"]]["labels"].to_numpy()   # the ranking of the search engine
        total_relevant = r["labels"].sum()                  # all the relevant judged documents
        per_query[q_id] = evaluate_query(returned, K, total_relevant)
    table = pd.DataFrame(per_query).T.drop(columns="RR@{}".format(K))   # RR@K is only used for the MRR
    means = {name: float(np.round(value, 3)) for name, value in mean_metrics(list(per_query.values()), K).items()}
    return table.astype(float).round(3), means

validation_table, validation_means = evaluation_table(validation_rankings)
print(validation_means)
validation_table

{'MAP@10': 0.669, 'MRR@10': 1.0}


,P@10,R@10,AP@10,F1@10,NDCG@10
1,1.000,0.769,0.769,0.870,1.000
2,0.600,0.600,0.569,0.600,0.716


In [12]:
# ranking of the judged documents (to check the metrics by hand)
for q_id, r in validation_rankings.items():
    print("Query {}: labels of the returned documents in ranking order = {}".format(q_id, list(r[r["retrieved"]]["labels"])))

Query 1: labels of the returned documents in ranking order = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0]
Query 2: labels of the returned documents in ranking order = [1, 1, 1, 1, 0, 1, 1, 0]


### 9 - Our test queries: relevance judgments

We acted as expert judges: for each of our five queries we labelled the **top 20** documents returned by the
search engine with 1 (relevant) or 0 (non-relevant). A document is relevant only if it matches **everything**
the query asks for (product type, gender, color, pattern, neck type, material...). The labels are in
`own_queries_labels.csv`. Since only the top 20 are judged, the recall is computed with respect to the
relevant documents found in these 20.

In [13]:
own_labels = pd.read_csv(OWN_LABELS_PATH)

# check that the labelled documents are exactly the current top 20 of each query
for q_id, query in own_queries.items():
    top20 = [pid for pid, score in results[q_id][:20]]
    assert list(own_labels[own_labels["query_id"] == q_id].sort_values("rank")["pid"]) == top20

own_labels.groupby(["query_id", "query"])["labels"].agg(["sum", "count"]).rename(columns={"sum": "relevant", "count": "judged"})

,,relevant,judged
query_id,query,,
3,women black round neck tshirt,20,20
4,men checkered casual cotton shirt,20,20
5,women printed kurta,13,20
6,men white sneakers,1,20
7,women v neck tshirt,1,20


In [14]:
own_rankings = {}
for q_id in own_queries:
    own_rankings[q_id] = own_labels[own_labels["query_id"] == q_id].sort_values("rank").reset_index(drop=True)
    own_rankings[q_id]["retrieved"] = True   # all of them are returned by the search engine

own_table, own_means = evaluation_table(own_rankings)
print(own_means)
own_table

{'MAP@10': 0.331, 'MRR@10': 0.65}


,P@10,R@10,AP@10,F1@10,NDCG@10
3,1.000,0.500,0.500,0.667,1.000
4,1.000,0.500,0.500,0.667,1.000
5,0.600,0.462,0.403,0.522,0.696
6,0.000,0.000,0.000,0.000,0.000
7,0.100,1.000,0.250,0.182,0.431


In [15]:
# non-relevant documents in the top 10 of the harder queries
for q_id in [6, 7]:
    r = own_rankings[q_id]
    print("Query {}: '{}'".format(q_id, own_queries[q_id]))
    for _, row in r[(r["labels"] == 0) & (r["rank"] <= 10)].iterrows():
        print("   rank {}: {}".format(row["rank"], row["title"]))

Query 6: 'men white sneakers'
   rank 1: Solid Men Round Neck White T-Shirt
   rank 2: Printed Men Round Neck White T-Shirt
   rank 3: Solid Men Collared Neck Yellow, White T-Shirt  (Pack of 2)
   rank 4: Solid Men Collared Neck White, Orange T-Shirt  (Pack of 2)
   rank 5: Solid Men Collared Neck White, Orange T-Shirt  (Pack of 2)
   rank 6: Solid Men V-neck Blue T-Shirt
   rank 7: Printed Men Round Neck White T-Shirt
   rank 8: Checkered Men Polo Neck White T-Shirt
   rank 9: Sneakers For Women  (White)
   rank 10: Men Regular Fit Solid Button Down Collar Casual Shirt
Query 7: 'women v neck tshirt'
   rank 1: Solid Women Round Neck Black T-Shirt
   rank 2: Solid Women Round Neck White T-Shirt
   rank 3: Solid Women Round Neck White T-Shirt
   rank 5: Solid Women Round Neck Black T-Shirt
   rank 6: Solid Women Round Neck Blue T-Shirt
   rank 7: Solid Women Round Neck Black T-Shirt
   rank 8: Solid Women Round Neck Black T-Shirt
   rank 9: Printed Women Round Neck Blue T-Shirt
   rank 

### 10 - Some problems of the system (used in the analysis of the report)

In [16]:
# 1) Conjunctive queries are too strict: relevant judged documents of query 2 that are not returned
r = validation_rankings[2]
missed = r[(r["labels"] == 1) & (~r["retrieved"])]["pid"]
for pid in missed:
    print("Not returned (query 2):", corpus[pid]["meta"]["title"], "| missing terms:",
          [t for t in build_terms("men slim jeans blue") if t not in corpus[pid]["search_tokens"]])

Not returned (query 2): Slim Women Blue Jeans | missing terms: ['men']
Not returned (query 2): Tapered Fit Men Blue Jeans | missing terms: ['slim']
Not returned (query 2): Super Skinny Women Blue Jeans | missing terms: ['men']
Not returned (query 2): Super Skinny Women Blue Jeans | missing terms: ['men']


In [17]:
# 2) Words that disappear in the pre-processing: the "v" of "v neck" is a one-character token
print(build_terms("women v neck tshirt"))

# 3) Brand names are cut in the corpus ("REEB"): "reebok" only appears in the description of some products
reeb = [pid for pid, doc in corpus.items() if doc["meta"]["brand"] == "REEB"]
print("REEB products: {} | with the term 'reebok': {}".format(len(reeb), sum("reebok" in corpus[pid]["search_tokens"] for pid in reeb)))
print("Results for 'reebok women track suit': {}".format(len(search_tf_idf("reebok women track suit", index, weights, idf, doc_length))))
print("REEB women track suits in the corpus:", [corpus[pid]["meta"]["title"] for pid in reeb if "Women Track Suit" in corpus[pid]["meta"]["title"]])

# 4) "men white sneakers": the term "sneaker" also appears in descriptions of t-shirts ("pair it with sneakers"),
#    and the sub_category "Men's Footwear" also contains women's shoes, so they match "men"
res = results[6]
shoes = [rank + 1 for rank, (pid, score) in enumerate(res) if "Sneakers" in corpus[pid]["meta"]["title"]]
print("\nResults of 'men white sneakers': {} | real sneakers: {} at ranks {}".format(len(res), len(shoes), shoes))
pid = res[0][0]
print("First result:", corpus[pid]["meta"]["title"], "| description:", corpus[pid]["meta"]["description"][:120])
for rank in shoes:
    pid = res[rank - 1][0]
    print("rank {}: {} | sub_category: {}".format(rank, corpus[pid]["meta"]["title"], corpus[pid]["meta"]["sub_category"]))
footwear = [doc for doc in corpus.values() if doc["meta"]["sub_category"] == "Men's Footwear"]
print("'Men's Footwear' products: {} | Ideal For = Women: {}".format(
    len(footwear), sum(doc["meta"]["product_details"].get("Ideal For") == "Women" for doc in footwear)))

['women', 'neck', 'tshirt']
REEB products: 802 | with the term 'reebok': 159
Results for 'reebok women track suit': 0
REEB women track suits in the corpus: ['Solid Women Track Suit', 'Solid Women Track Suit', 'Solid Women Track Suit', 'Solid Women Track Suit']

Results of 'men white sneakers': 66 | real sneakers: 5 at ranks [9, 12, 27, 38, 56]
First result: Solid Men Round Neck White T-Shirt | description: White solid T-shirt, has a round neck, and . Made of cotton, youll love the high-quality fit and feel of this Levis tee.
rank 9: Sneakers For Women  (White) | sub_category: Men's Footwear
rank 12: Sneakers For Men  (White) | sub_category: Men's Footwear
rank 27: Sneakers For Men  (White) | sub_category: Men's Footwear
rank 38: Sneakers For Women  (White) | sub_category: Men's Footwear
rank 56: Sneakers For Women  (White) | sub_category: Men's Footwear
'Men's Footwear' products: 920 | Ideal For = Women: 182


In [18]:
# 5) Many documents get exactly the same score (same title and attributes): the order among them is arbitrary
for q_id, query in own_queries.items():
    top_scores = [np.round(s, 6) for pid, s in results[q_id][:20]]
    print("Query {}: different scores in the top 20 = {}".format(q_id, len(set(top_scores))))

Query 3: different scores in the top 20 = 15
Query 4: different scores in the top 20 = 5
Query 5: different scores in the top 20 = 18
Query 6: different scores in the top 20 = 19
Query 7: different scores in the top 20 = 14


In [19]:
# 6) The cosine divides by the length |d| of the document, so short documents (usually without description)
#    get higher scores than long ones that contain the same query terms
for q_id, query in own_queries.items():
    res = results[q_id]
    lengths = [len(corpus[pid]["search_tokens"]) for pid, score in res]
    empty = [corpus[pid]["meta"]["description"] == "" for pid, score in res]
    print("Query {}: mean terms top 10 = {} vs all results = {} | empty description top 10 = {}% vs all = {}%".format(
        q_id, np.round(np.mean(lengths[:10]), 1), np.round(np.mean(lengths), 1),
        int(np.round(100 * np.mean(empty[:10]))), int(np.round(100 * np.mean(empty)))))

Query 3: mean terms top 10 = 23.8 vs all results = 51.6 | empty description top 10 = 100% vs all = 31%
Query 4: mean terms top 10 = 18.3 vs all results = 32.0 | empty description top 10 = 100% vs all = 42%
Query 5: mean terms top 10 = 31.2 vs all results = 62.2 | empty description top 10 = 80% vs all = 33%
Query 6: mean terms top 10 = 51.1 vs all results = 73.4 | empty description top 10 = 10% vs all = 8%
Query 7: mean terms top 10 = 23.6 vs all results = 51.4 | empty description top 10 = 100% vs all = 33%
